In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('Harga Mobil Bekas.csv')
display(df.head())

In [ ]:
print(df.isnull().sum())

In [ ]:
df_clean = df.drop(columns=['Unnamed: 0', 'Name', 'New_Price'])

def clean_numeric(val):
    if pd.isna(val):
        return np.nan
    try:
        return float(str(val).split()[0])
    except:
        return np.nan

df_clean['Mileage'] = df_clean['Mileage'].apply(clean_numeric)
df_clean['Engine'] = df_clean['Engine'].apply(clean_numeric)
df_clean['Power'] = df_clean['Power'].replace('null bhp', np.nan)
df_clean['Power'] = df_clean['Power'].apply(clean_numeric)

for col in ['Mileage', 'Engine', 'Power', 'Seats']:
    df_clean[col].fillna(df_clean[col].median(), inplace=True)

display(df_clean.head())

In [ ]:
plt.figure(figsize=(6, 6))
df_clean['Transmission'].value_counts().plot.pie(autopct='%1.1f%%', colors=['#4C72B0', '#55A868'], explode=(0, 0.1), shadow=True)
plt.title('Proporsi Transmisi (Manual vs Automatic)')
plt.ylabel('')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='Transmission', y='Engine', data=df_clean, palette='Set2')
plt.title('Perbandingan Kapasitas Mesin (Engine) Berdasarkan Transmisi')
plt.show()

In [ ]:
sns.pairplot(df_clean[['Kilometers_Driven', 'Mileage', 'Engine', 'Power', 'Transmission']], hue='Transmission', diag_kind='kde', palette='husl')
plt.show()

In [ ]:
categorical_cols = ['Location', 'Fuel_Type', 'Owner_Type']
df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)

le = LabelEncoder()
df_encoded['Transmission'] = le.fit_transform(df_encoded['Transmission'])

X = df_encoded.drop('Transmission', axis=1)
y = df_encoded['Transmission']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

y_pred = rf_model.predict(X_test_scaled)
y_prob = rf_model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix - Random Forest')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc_score:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

In [ ]:
feature_importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importances[:10].values, y=feature_importances[:10].index, palette='viridis')
plt.title('Top 10 Feature Importances')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.show()